# MD 候选分子走查 · A2A 腺苷受体拮抗剂智能虚拟筛选

赛道三（离子通道、GPCR 等药靶的小分子药物虚拟筛选）。本 Notebook 按赛事《代码提交要求》
附件 5 §二(4) 提供，用于展示本项目的关键流程**可复现**：

1. 读取三条链的 MD 候选清单（目录化合物 / ML 生成分子 / 二者联立合并）
2. 复算批次读数：来源构成、CNN_VS 分位覆盖、最优性证书
3. 复核 hERG 双模型分歧标注
4. 调用主运行入口 `predict.py` 生成标准化候选清单，并与冻结清单逐行核验

> 只读随包数据，不需要外网与上游大文件。

In [ ]:
import json, subprocess, sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("包根：", ROOT.name)
print("python", sys.version.split()[0], "| pandas", pd.__version__)

SETS = {
    "SOP_5": "results/SOP_5/SOP-05_MD_batch20.csv",
    "SOP_ML_run05": "results/SOP_ML_run05/SOP-05_MD_batch20.csv",
    "merged_final_md": "results/merged_final_md/final_md_20.csv",
}
tables = {}
for k, rel in SETS.items():
    d = pd.read_csv(ROOT / rel)
    tables[k] = d
    cand = int((d.candidate_type == "candidate").sum())
    print(f"{k:16s} {len(d):2d} 行 = 候选 {cand:2d} + 对照 {len(d)-cand}   ← {rel}")

## 1. 三套清单与联立合并层的读数

合并层把两条链的 MD 一级批次并成 35 个候选池，通用骨架去重后资格池 33 个，
在其中**重解**出 20 席。下面直接读它的自描述 JSON（含最优性证书与输入 sha256）。

In [ ]:
rep = json.loads((ROOT / "results/merged_final_md/final_md.json").read_text(encoding="utf-8"))
opt = rep["optimality"]
print("合并候选池：", rep["merge"]["n_candidates"], "| 资格池：", rep["eligibility"]["n_pool"])
print("名册来源构成：", rep["roster"]["source_counts"])
for st in opt["journal"]:
    print(f"  阶段{st['stage']} {st['name']:24s} 求解目标={st['objective']:.6f} "
          f"status={st['status']} mip_gap={st['mip_gap']}")
print("t* =", opt["t_star"], "| 跨度 =", opt["span"], "| Σ CNN_VS =", opt["sum_CNN_VS"])

## 2. CNN_VS 分位覆盖与约束实测（名册 vs 目标）

分位按**资格池自分布**切 3 轴 × 4 层 = 12 格；覆盖下限 `t*` 是第一阶段目标。

In [ ]:
roster = tables["merged_final_md"]
cand = roster[roster.candidate_type == "candidate"].copy()
cand["src"] = cand.source_branch.str.contains("生成").map({True: "ML 生成", False: "目录"})
print("名册候选数：", len(cand), "| 来源构成：", cand.src.value_counts().to_dict())
print("CNN_VS 跨度：", round(cand.CNN_VS.max() - cand.CNN_VS.min(), 4),
      "（端点 %.4f / %.4f）" % (cand.CNN_VS.max(), cand.CNN_VS.min()))
print("Σ CNN_VS：", round(cand.CNN_VS.sum(), 4))
print("批内成对 ECFP4 Tc 上限：", rep["optimality"]["compliance"]["max_pairwise_tc"],
      "（阈值 0.65）")
print("\n分位格覆盖（CNN_VS × cLogP × CNS MPO）：")
print(cand.coverage_bins.value_counts().head(12).to_string())

## 3. hERG 双模型分歧（**只表示「在 ADMET-AI 口径下过门」**）

主模型 = ADMET-AI；第二模型 = 本项目自建的 ChEMBL ECFP4 集成。两者一致性接近随机，
分歧分子**保留入池并逐分子标注「模型依赖」**。

In [ ]:
n_flag = int((cand.herg_cross_model_flag == "cross_model_flag").sum()) if \
        "cross_model_flag" in cand.columns else int((cand.md_eligibility_note.fillna("") != "").sum())
print(f"名册内第二模型分歧（标注「模型依赖」）候选：{n_flag} / {len(cand)}")
pd.set_option("display.width", 200)
cols = ["cid", "CNN_VS", "herg_prob", "hERG_cross_ens", "md_eligibility_note"]
print(cand[cols].sort_values("hERG_cross_ens", ascending=False).head(5).to_string(index=False))

## 4. 主运行入口：生成标准化候选清单

`predict.py` 把三条来源的清单归一为附件 5 §四要求的候选清单（候选编号 / 赛道 / SMILES /
关键预测指标 / 模型与运行版本 / 备注）。

In [ ]:
out = subprocess.run([sys.executable, "predict.py", "--source", "frozen"],
                     cwd=ROOT, capture_output=True, text=True)
print(out.stdout[-1200:])
assert out.returncode == 0, out.stderr
unified = pd.read_csv(ROOT / "results/results.csv")
print("统一清单：", unified.shape)
print(unified.groupby("source_set").agg(行数=("candidate_id", "size"),
                                        候选=("role", lambda s: (s == "候选").sum()),
                                        唯一分子=("molecule_id", "nunique")))

## 5. 一致性核验（复跑 vs 冻结）

`run.sh` 会用随包中间产物复跑选择层与合并层，再用 `predict.py --verify` 与冻结清单逐行比对。
本 Notebook 先核验现有 `results/reproduced/`（若已跑过 `./run.sh`）。

In [ ]:
rep_csv = ROOT / "results/reproduced/results.csv"
if rep_csv.is_file():
    v = subprocess.run([sys.executable, "predict.py", "--verify", str(rep_csv),
                        str(ROOT / "results/results.csv")], cwd=ROOT, capture_output=True, text=True)
    print(v.stdout.strip() or v.stderr.strip())
else:
    print("尚未跑过 ./run.sh —— 在包根执行 ./run.sh 后用本单元核验。")

## 6. 结论与边界

- 三套清单共用同一冻结规则 `SOP-05-A2A-antagonist-shortlist v1.7.0`；
  两条链的 MD 一级批次结构零重叠，合并层是「两个互斥来源取并集后重解」。
- 读数（`t*`、跨度、Σ CNN_VS）只说明**在该目标函数口径下最优**，不是「比落选者更好」。
- hERG 只表示「在 ADMET-AI 口径下过门」；第二模型分歧分子已保留并标注，正交出口是膜片钳。
- 跨来源 CNN_VS 不可直接比较（选池口径不同），合并层已给出百分位口径的敏感性对照。
- 本清单是计算结果，**不含实验数据**，不产生安全性或有效性结论。